# 03 — Introduction à l'entraînement

**Objectif pédagogique** : comprendre ce que fait *concrètement* la boucle d'entraînement d'un U-Net de débruitage, et pouvoir lancer un mini-entraînement de quelques minutes pour voir la loss descendre.

**Ce notebook tourne en local sur CPU** : il utilise un sous-ensemble de ~50 paires audio. Une fois compris, on passe au [notebook 04](04_training_session.ipynb) pour la grosse session GPU sur Colab.

**Pré-requis** : le dépôt installé (`pip install -r requirements.txt`) et les dossiers `data/train/{clean,noisy}` et `data/val/{clean,noisy}` peuplés (au moins quelques paires).

## 1. Pourquoi un entraînement ?

Le U-Net défini dans [`src/model.py`](../src/model.py) a ≈ 7,85 M paramètres initialisés au hasard. Brut, il ne sait pas débruiter — il sort un masque aléatoire.

**Entraîner** = ajuster ces paramètres pour que, sur le maximum de paires `(noisy_mag, clean_mag)` du dataset, la sortie du réseau soit aussi proche que possible de `clean_mag`.

Le mécanisme est toujours le même :
1. on présente un *batch* (paquet) de paires au modèle ;
2. on calcule une **loss** : un nombre qui mesure l'écart entre prédiction et cible ;
3. on calcule le *gradient* de cette loss par rapport à chaque paramètre (`.backward()`) ;
4. l'**optimizer** met à jour les paramètres dans la direction qui fait baisser la loss (`.step()`).

On répète des milliers de fois. C'est tout.

In [ ]:
# Setup : on remonte d'un cran pour importer src/ et on fixe la graine.
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src import config as cfg
from src.model import UNet, count_parameters
from src.dataset import PairedAudioDataset
from src.metrics import magnitude_mse_loss, si_sdr_mag, mask_statistics, OverfitMonitor

torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device :", device)
print("params U-Net (base=32) :", f"{count_parameters(UNet(base_channels=32)):,}")

## 2. Anatomie d'un pas d'entraînement

On va exécuter **manuellement** les 4 étapes ci-dessus sur un seul batch pour bien voir ce qui se passe. On utilise volontairement un modèle plus petit (`base_channels=16`) pour aller vite en CPU.

In [ ]:
# Adapter ces chemins si besoin pour le test local.
# Sur Colab/Drive, src.config pointe déjà au bon endroit.
NOISY_DIR = cfg.DATA_TRAIN_NOISY
CLEAN_DIR = cfg.DATA_TRAIN_CLEAN

ds = PairedAudioDataset(noisy_dir=NOISY_DIR, clean_dir=CLEAN_DIR, crop_mode="random")
print("paires disponibles :", len(ds))

from torch.utils.data import DataLoader
loader = DataLoader(ds, batch_size=4, shuffle=True)
batch = next(iter(loader))
print("noisy_mag :", batch["noisy_mag"].shape)   # (B, 1, F, T)
print("clean_mag :", batch["clean_mag"].shape)

In [ ]:
# 1. instancier modèle + optimizer
model = UNet(base_channels=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

noisy = batch["noisy_mag"].to(device)
clean = batch["clean_mag"].to(device)

# 2. forward — le réseau prédit une magnitude propre
pred = model(noisy)
print("pred :", pred.shape, " min/max :", float(pred.min()), float(pred.max()))

# 3. loss — un seul scalaire
loss = magnitude_mse_loss(pred, clean)
print("loss avant step :", float(loss))

# 4. backward + step — c'est là que les poids bougent
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()

# refaire le forward pour vérifier que la loss a un peu bougé
with torch.no_grad():
    new_loss = magnitude_mse_loss(model(noisy), clean)
print("loss après step :", float(new_loss))

Sur **un seul** pas, la loss bouge un peu. Ce n'est pas significatif (le batch suivant, choisi au hasard, donnera peut-être une loss plus haute). Ce qui compte, c'est la **tendance** sur des dizaines, centaines, milliers de pas.

## 3. Pourquoi la MSE sur magnitude ?

On compare élément par élément `pred_mag` et `clean_mag` (forme `(B, 1, F, T)`) avec une *Mean Squared Error*. C'est :

- **simple** : un seul nombre, dérivable partout ;
- **stable** : pas de bornes molles ni de logs qui explosent en zéro ;
- **physiquement défendable** : si la magnitude prédite est proche de la magnitude propre, le son reconstruit le sera aussi (à la phase près, qu'on reprend du signal bruité).

On verra en semaine 4 des alternatives (L1, log-magnitude, loss spectrale) — mais commencer simple permet d'avoir une **baseline** propre.

## 4. Pourquoi Adam et `lr=1e-3` ?

Le *learning rate* (LR) règle la taille des pas que fait l'optimizer. Trop grand → la loss diverge ou oscille. Trop petit → l'entraînement n'avance pas dans un temps raisonnable.

**Adam** adapte automatiquement un LR par paramètre à partir de l'historique des gradients. `1e-3` est sa valeur par défaut quasi-universelle et marche bien sur ce type de tâche. On la garde, et on laisse un `ReduceLROnPlateau` baisser le LR automatiquement si la val_loss stagne (voir [`src/train.py`](../src/train.py)).

## 5. Mini-entraînement (≈ 5 minutes en CPU)

On appelle la fonction `train` de [`src/train.py`](../src/train.py) en limitant fortement les données :
- 50 paires d'entraînement, 10 paires de validation ;
- 10 epochs ;
- `base_channels=16` (modèle plus léger).

C'est juste assez pour **voir la loss descendre**. Pas pour un modèle utile.

In [ ]:
from src.train import train

mini_config = {
    "run_id":              "mini-cpu-demo",
    "lr":                  1e-3,
    "batch_size":          4,
    "num_workers":         0,        # 0 = pas de multiprocessing (plus simple en notebook)
    "num_epochs":          10,
    "base_channels":       16,
    "use_wandb":           False,    # local : on ne logge pas en ligne
    "early_stop_enabled":  False,    # 10 epochs, pas la peine
    "ckpt_every_n_epochs": 5,
}

history = train(
    mini_config,
    max_train_samples=50,
    max_val_samples=10,
)

## 6. Lire les courbes

Le dict `history` contient une liste par métrique, alignée sur les epochs. On affiche les deux courbes les plus parlantes : `train_loss` et `val_loss`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["epoch"], history["train_loss"], label="train", marker="o")
ax.plot(history["epoch"], history["val_loss"],   label="val",   marker="s")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE")
ax.set_title("Mini-entraînement — loss MSE")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

### Comment lire ça

Trois scénarios typiques :

1. **Train ↘ et val ↘ ensemble** (l'idéal) — le modèle apprend et généralise. C'est ce qu'on espère voir.
2. **Train ↘ et val plate** — le modèle apprend, mais sur si peu de données qu'il ne généralise pas. Normal sur 50 paires : ce n'est qu'un sanity check.
3. **Train ↘ et val ↗** — *overfitting* franc. À surveiller en grosse session.

Le notebook 05 (`05_training_analysis.ipynb`) gère tout cela proprement, avec courbes et flags d'overfit lus directement depuis le JSONL.

## 7. À retenir

- Toujours **fixer la seed** (`torch.manual_seed`, `np.random.seed`) pour pouvoir comparer deux runs.
- Vérifier le **device** (`cuda` vs `cpu`) avant de lancer.
- L'`history` retourné par `train()` est **aussi sauvegardé sur disque** dans `logs/<run_id>/history.jsonl` — utile pour l'analyse a posteriori.
- Pour la vraie session d'entraînement, voir [04_training_session.ipynb](04_training_session.ipynb).